In [0]:
dbutils.widgets.removeAll()

In [0]:
# Databricks notebook source
# DBTITLE 1, Setup Logging & Imports
import logging
from datetime import datetime
import os
from pyspark.sql.functions import col, sha2, concat_ws, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
# Logging setup
base_dir = os.getcwd()
log_dir = os.path.abspath(os.path.join(base_dir, "../../logs"))
os.makedirs(log_dir, exist_ok=True)
log_filename = f"{log_dir}/person_silver_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename.replace('file:', '')),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info("=" * 60)
logger.info("Silver Person Processing Started")
logger.info("=" * 60)

In [0]:
# COMMAND ----------
# DBTITLE 1, Configuration - Source & Target Tables
sourcePath = "/Volumes/claimspan/bronze/member_consolidated"
silverPersonBridgeTable = "claimspan.silver.silver_memberpersonbridge"
silverPersonTable = "claimspan.silver.silver_member"

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Source Path: {sourcePath}")
print(f"Silver Person Bridge Table: {silverPersonBridgeTable}")
print(f"Silver Person Table: {silverPersonTable}")
print("="*60)

logger.info("Configuration loaded:")
logger.info(f"  Source Path: {sourcePath}")
logger.info(f"  Silver Person Bridge Table: {silverPersonBridgeTable}")
logger.info(f"  Silver Person Table: {silverPersonTable}")

In [0]:
# COMMAND ----------
# DBTITLE 1, Helper Functions
def table_exists(tableToCheck):
    """Check if a Unity Catalog table or path exists"""
    try:
        if tableToCheck.startswith("/"):
            spark.read.format("delta").load(tableToCheck)
        else:
            spark.table(tableToCheck)
        logger.info(f"Table/Path check: {tableToCheck} - Exists")
        return True
    except Exception as e:
        logger.warning(f"Table/Path check: {tableToCheck} - Does not exist or inaccessible: {str(e)}")
        return False

In [0]:
# COMMAND ----------
# DBTITLE 1, Define SQL Transformations

srcsql = """
WITH consolidateMem1 AS (
  SELECT 
    brdg.ESAIInternalPersonID, 
    brdg.UniqueRecord, 
    mem.ClientID, 
    mem.FileID, 
    mem.LoadDateTime, 
    mem.FileLayoutID, 
    mem.FileLayoutDescription,
    -- FHIR Identifiers
    UPPER(TRIM(mem.identifier_uniquepersonkey)) as identifier_uniquepersonkey,
    UPPER(TRIM(mem.identifier_planMemberID)) as identifier_planMemberID,
    UPPER(TRIM(mem.identifier_subscriberID)) as identifier_subscriberID,
    UPPER(TRIM(mem.identifier_beneficiaryID)) as identifier_beneficiaryID,
    UPPER(TRIM(mem.identifier_enrolleeUniqueID)) as identifier_enrolleeUniqueID,
    -- FHIR Name fields
    UPPER(TRIM(mem.name_family)) as name_family,
    UPPER(TRIM(mem.name_given_first)) as name_given_first,
    UPPER(TRIM(mem.name_given_middle)) as name_given_middle,
    UPPER(TRIM(mem.name_prefix)) as name_prefix,
    UPPER(TRIM(mem.name_suffix)) as name_suffix,
    mem.name_text as name_text,
    -- Dates & Demographics
    TRY_CAST(mem.birthDate AS STRING) as birthDate,
    TRY_CAST(mem.deceasedDateTime AS STRING) as deceasedDateTime,
    UPPER(TRIM(mem.gender)) as gender,
    -- Permanent Address
    UPPER(TRIM(mem.address_permanent_line1)) as address_permanent_line1,
    UPPER(TRIM(mem.address_permanent_line2)) as address_permanent_line2,
    UPPER(TRIM(mem.address_permanent_city)) as address_permanent_city,
    UPPER(TRIM(mem.address_permanent_district)) as address_permanent_district,
    UPPER(TRIM(mem.address_permanent_state)) as address_permanent_state,
    UPPER(TRIM(mem.address_permanent_postalCode)) as address_permanent_postalCode,
    -- Mailing Address
    UPPER(TRIM(mem.address_mailing_line1)) as address_mailing_line1,
    UPPER(TRIM(mem.address_mailing_line2)) as address_mailing_line2,
    UPPER(TRIM(mem.address_mailing_city)) as address_mailing_city,
    UPPER(TRIM(mem.address_mailing_state)) as address_mailing_state,
    UPPER(TRIM(mem.address_mailing_postalCode)) as address_mailing_postalCode,
    UPPER(TRIM(mem.address_mailing_district)) as address_mailing_district,
    -- Telecom
    TRIM(mem.telecom_phone_home) as telecom_phone_home,
    TRIM(mem.telecom_email) as telecom_email,
    TRIM(mem.telecom_fax) as telecom_fax,
    -- Additional Identifiers & Extensions
    UPPER(TRIM(mem.identifier_medicaidID)) as identifier_medicaidID,
    UPPER(TRIM(mem.extension_maskedMemberID)) as extension_maskedMemberID,
    UPPER(TRIM(mem.extension_race_text)) as extension_race_text,
    UPPER(TRIM(mem.extension_race_dataSource)) as extension_race_dataSource,
    UPPER(TRIM(mem.extension_ethnicity_ombCategory_code)) as extension_ethnicity_ombCategory_code,
    UPPER(TRIM(mem.extension_ethnicity_dataSource)) as extension_ethnicity_dataSource,
    -- Communication
    UPPER(TRIM(mem.communication_spokenLanguage_text)) as communication_spokenLanguage_text,
    UPPER(TRIM(mem.communication_spokenLanguage_codeSystem)) as communication_spokenLanguage_codeSystem,
    UPPER(TRIM(mem.communication_writtenLanguage_code)) as communication_writtenLanguage_code,
    UPPER(TRIM(mem.communication_writtenLanguage_codeSystem)) as communication_writtenLanguage_codeSystem,
    UPPER(TRIM(mem.communication_otherLanguage_text)) as communication_otherLanguage_text,
    UPPER(TRIM(mem.communication_otherLanguage_codeSystem)) as communication_otherLanguage_codeSystem,
    -- Contact Caretaker
    UPPER(TRIM(mem.contact_caretaker_name_given_first)) as contact_caretaker_name_given_first,
    UPPER(TRIM(mem.contact_caretaker_name_family)) as contact_caretaker_name_family,
    UPPER(TRIM(mem.contact_caretaker_name_given_middle)) as contact_caretaker_name_given_middle,
    -- Employment & Status
    UPPER(TRIM(mem.extension_usCitizenStatus)) as extension_usCitizenStatus,
    UPPER(TRIM(mem.extension_enrolleeEducation)) as extension_enrolleeEducation,
    UPPER(TRIM(mem.extension_enrolleeEmployment)) as extension_enrolleeEmployment,
    UPPER(TRIM(mem.extension_coverageProduct_id)) as extension_coverageProduct_id,
    -- Alternate Keys
    mem.identifier_alternateKey1 as identifier_alternateKey1,
    mem.identifier_alternateKey2 as identifier_alternateKey2,
    mem.identifier_alternateKey3 as identifier_alternateKey3,
    mem.identifier_alternateKey4 as identifier_alternateKey4,
    mem.identifier_alternateKey5 as identifier_alternateKey5,
    mem.identifier_alternateKey6 as identifier_alternateKey6,
    mem.identifier_alternateKey7 as identifier_alternateKey7,
    mem.identifier_alternateKey8 as identifier_alternateKey8,
    mem.identifier_alternateKey9 as identifier_alternateKey9,
    mem.identifier_alternateKey10 as identifier_alternateKey10,
    -- Bridge Metadata
    brdg.PMUP, 
    brdg.IsCurrentPMUP
  FROM consolidateMem mem
  INNER JOIN silverPersonBrdg brdg 
    ON mem.UniqueRecord = brdg.UniqueRecord 
    AND mem.FileLayoutID = brdg.FileLayoutID 
    AND brdg.IsCurrentPMUP = 1
)
SELECT 
  ESAIInternalPersonID, UniqueRecord, ClientID, FileID, LoadDateTime, FileLayoutID, 
  FileLayoutDescription, identifier_uniquepersonkey, identifier_planMemberID, identifier_subscriberID, identifier_beneficiaryID, identifier_enrolleeUniqueID,
  name_family, name_given_first, name_given_middle, birthDate, deceasedDateTime, 
  gender, address_permanent_line1, address_permanent_line2, address_permanent_city, address_permanent_district, 
  address_permanent_state, address_permanent_postalCode, address_mailing_line1, address_mailing_line2, 
  address_mailing_city, address_mailing_state, address_mailing_postalCode, address_mailing_district, telecom_phone_home, telecom_email, 
  identifier_medicaidID, telecom_fax, extension_race_text, extension_race_dataSource, contact_caretaker_name_given_first, contact_caretaker_name_family, 
  contact_caretaker_name_given_middle, extension_ethnicity_ombCategory_code, extension_ethnicity_dataSource, communication_spokenLanguage_text, 
  communication_spokenLanguage_codeSystem, communication_writtenLanguage_code, communication_writtenLanguage_codeSystem, 
  communication_otherLanguage_text, communication_otherLanguage_codeSystem, extension_usCitizenStatus, identifier_alternateKey1, identifier_alternateKey2, 
  identifier_alternateKey3, identifier_alternateKey4, identifier_alternateKey5, identifier_alternateKey6, identifier_alternateKey7, 
  identifier_alternateKey8, identifier_alternateKey9, identifier_alternateKey10, extension_maskedMemberID, extension_enrolleeEducation, 
  extension_enrolleeEmployment, PMUP, IsCurrentPMUP, extension_coverageProduct_id, name_prefix, name_suffix, name_text,
  sha2(concat(
    IfNull(ESAIInternalPersonID,""), "|", IfNull(UniqueRecord,""), "|", 
    IfNull(ClientID,""), "|", IfNull(CAST(FileID AS STRING),""), "|", 
    IfNull(CAST(LoadDateTime AS STRING),""), "|", IfNull(CAST(FileLayoutID AS STRING),""), "|", 
    IfNull(FileLayoutDescription,""), "|", IfNull(identifier_uniquepersonkey,""), "|", 
    IfNull(identifier_planMemberID,""), "|", IfNull(identifier_subscriberID,""), "|", 
    IfNull(identifier_beneficiaryID,""), "|", 
    IfNull(identifier_enrolleeUniqueID, ""), "|",
    IfNull(name_family,""), "|", IfNull(name_given_first,""), "|", 
    IfNull(name_given_middle,""), "|", 
    IfNull(CAST(birthDate AS STRING),""), "|", IfNull(CAST(deceasedDateTime AS STRING),""), "|", 
    IfNull(gender,""), "|", IfNull(address_permanent_line1,""), "|", 
    IfNull(address_permanent_line2,""), "|", IfNull(address_permanent_city,""), "|", 
    IfNull(address_permanent_district,""), "|", IfNull(address_permanent_state,""), "|", 
    IfNull(address_permanent_postalCode,""), "|", IfNull(address_mailing_line1,""), "|", 
    IfNull(address_mailing_line2,""), "|", IfNull(address_mailing_city,""), "|", 
    IfNull(address_mailing_state,""), "|", IfNull(address_mailing_postalCode,""), "|", 
    IfNull(address_mailing_district,""), "|", IfNull(telecom_phone_home,""), "|", IfNull(telecom_email,""), "|", 
    IfNull(identifier_medicaidID,""), "|", IfNull(telecom_fax,""), "|", IfNull(extension_race_text,""), "|", 
    IfNull(extension_race_dataSource,""), "|", IfNull(contact_caretaker_name_given_first,""), "|", 
    IfNull(contact_caretaker_name_family,""), "|", IfNull(contact_caretaker_name_given_middle,""), "|", 
    IfNull(extension_ethnicity_ombCategory_code,""), "|", IfNull(extension_ethnicity_dataSource,""), "|", 
    IfNull(communication_spokenLanguage_text,""), "|", IfNull(communication_spokenLanguage_codeSystem,""), "|", 
    IfNull(communication_writtenLanguage_code,""), "|", IfNull(communication_writtenLanguage_codeSystem,""), "|", 
    IfNull(communication_otherLanguage_text,""), "|", IfNull(communication_otherLanguage_codeSystem,""), "|", 
    IfNull(extension_usCitizenStatus,""), "|", IfNull(identifier_alternateKey1,""), "|", IfNull(identifier_alternateKey2,""), "|", 
    IfNull(identifier_alternateKey3,""), "|", IfNull(identifier_alternateKey4,""), "|", 
    IfNull(identifier_alternateKey5,""), "|", IfNull(identifier_alternateKey6,""), "|", 
    IfNull(identifier_alternateKey7,""), "|", IfNull(identifier_alternateKey8,""), "|", 
    IfNull(identifier_alternateKey9,""), "|", IfNull(identifier_alternateKey10,""), "|", 
    IfNull(extension_maskedMemberID,""), "|", IfNull(extension_enrolleeEducation,""), "|", 
    IfNull(extension_enrolleeEmployment,""), "|", IfNull(PMUP,""), "|", 
    IfNull(CAST(IsCurrentPMUP AS STRING),""), "|", IfNull(extension_coverageProduct_id,""), "|", 
    IfNull(name_prefix,""), "|", IfNull(name_suffix,""), "|", IfNull(name_text,"")
  ), 256) AS HashKey 
FROM consolidateMem1
"""

In [0]:
# COMMAND ----------
# DBTITLE 1, Main Execution - Build Silver Person table

# Re-import to ensure col is the function, not a string variable
# from pyspark.sql.functions import col, sha2, concat_ws, row_number
# from pyspark.sql.window import Window

logger.info("=" * 60)
logger.info("MAIN EXECUTION STARTED")
logger.info("=" * 60)
print(f"\nProcessing Silver Person data...")
logger.info("Processing Silver Person data...")
print(f"Source: {sourcePath}")
print(f"Person Bridge: {silverPersonBridgeTable}")
logger.info(f"Source: {sourcePath}")
logger.info(f"Person Bridge: {silverPersonBridgeTable}")

logger.info(f"Checking if source path exists: {sourcePath}")
if table_exists(sourcePath):
    logger.info("Source table exists - proceeding with processing")
    
    # Load consolidated member data
    print("Loading consolidated member data...")
    logger.info("Loading consolidated member data from source path...")
    dfconsolidateMemSrc = spark.read.format("delta").load(sourcePath)
    source_count = dfconsolidateMemSrc.count()
    logger.info(f"Consolidated member data loaded: {source_count} records")
    
    # Load person bridge table
    print("Loading person bridge data...")
    logger.info(f"Loading person bridge table: {silverPersonBridgeTable}")
    dfsilverPersonBrdg = spark.table(silverPersonBridgeTable)
    bridge_count = dfsilverPersonBrdg.count()
    logger.info(f"Person bridge data loaded: {bridge_count} records")
    
    # Add UniqueRecord column to consolidated data (RESTORED OLD LOGIC)
    logger.info("Starting data transformations...")
    windowPartition = Window.partitionBy(col("FileId")).orderBy(col("RecordHash").desc())

    logger.info("Applying transformations: distinct, hash, row number, unique record")

    dfconsolidateMem = dfconsolidateMemSrc.distinct() \
        .withColumn("RecordHash", sha2(concat_ws("||", *[col(c) for c in dfconsolidateMemSrc.columns]), 256)) \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-", col("FileID"), col("RowNumber")))
    
    transformed_count = dfconsolidateMem.count()
    logger.info(f"Transformations completed: {transformed_count} records after distinct")
    
    # Create temp views
    logger.info("Creating temporary views for SQL processing")
    dfconsolidateMem.createOrReplaceTempView("consolidateMem")
    logger.info("Created temp view: consolidateMem")
    dfsilverPersonBrdg.createOrReplaceTempView("silverPersonBrdg")
    logger.info("Created temp view: silverPersonBrdg")
    
    # Join member data with person bridge
    print("Joining member data with person bridge...")
    logger.info("Executing SQL join between member and person bridge...")
    dfsrc = spark.sql(srcsql)
    logger.info("SQL join completed")
    
    recordCount = dfsrc.count()
    print(f"Found {recordCount} records to process")
    logger.info(f"Join result: {recordCount} records to process")
    
    if recordCount > 0:
        print(f"\nWriting/Merging to Silver Person table: {silverPersonTable}")
        logger.info(f"Processing {recordCount} records for Silver Person table: {silverPersonTable}")
        
        # Merge if target exists (Prevents duplicates on re-runs)
        if spark.catalog.tableExists(silverPersonTable):
            logger.info("Target table exists - executing MERGE to prevent duplicate records")
            targetTable = DeltaTable.forName(spark, silverPersonTable)
            targetTable.alias("target") \
                .merge(
                    dfsrc.alias("src"),
                    "target.UniqueRecord = src.UniqueRecord AND target.FileLayoutID = src.FileLayoutID AND target.HashKey = src.HashKey"
                ) \
                .whenNotMatchedInsertAll() \
                .execute()
        else:
            logger.info("Target table does not exist - creating initial table via append mode")
            dfsrc.write.format("delta").mode("append").saveAsTable(silverPersonTable)

        print("\n Silver Person table updated successfully!")
        logger.info("Silver Person table updated successfully")
        logger.info("=" * 60)
        logger.info("PROCESSING COMPLETED SUCCESSFULLY")
        logger.info("=" * 60)
    else:
        print("\n No records to process - check person bridge table has data")
        logger.warning("No records to process - join returned 0 records")
        logger.warning("Check if person bridge table has data with IsCurrentPMUP = 1")
        logger.info("=" * 60)
        logger.info("PROCESSING COMPLETED - NO RECORDS")
        logger.info("=" * 60)
else:
    print(f"\n Source path does not exist: {sourcePath}")
    logger.error(f"Source path does not exist: {sourcePath}")
    logger.info("=" * 60)
    logger.info("PROCESSING ABORTED - SOURCE PATH NOT FOUND")
    logger.info("=" * 60)

In [0]:
member_consolidated = spark.read.format('delta').load("/Volumes/claimspan/bronze/member_consolidated")
display(member_consolidated)